# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/glenkochwa-lang/flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*



**Task Type: Binary Classification mapped to a Scoring/Ranking queue.**

While the end goal is a ranked list for human reviewers, the underlying ML task is binary classification with probability scoring. Instead of just predicting a hard "1" or "0", we will use the model's predicted probability (e.g., an 85% confidence that a page is an underperformer) as a score. Sorting the dataset descending by this probability score gives us our ranked queue. This transforms a standard classification task into a prioritized decision-support tool.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# This cell is for CODE (numbers, a query, a check).
import pandas as pd
import numpy as np
import os

# Load data defensively just like Week 1
csv_path = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(csv_path):
    csv_path = os.path.join("..", "..", csv_path)
    
df = pd.read_csv(csv_path)
print(f"Base data loaded: {df.shape[0]} rows, {df.shape[1]} columns.")

Base data loaded: 30000 rows, 44 columns.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*



**Target: A defined rule (proxy label).**

Since we don't have historical ground-truth labels for "pages that improved when an editor fixed them," we must create a proxy label from observed behavior. Building on the Week 1 EDA, our target is a binary flag: `is_opportunity`.

A page gets a `1` if its CTR is less than 70% of the median CTR for its specific `position_tier`. It gets a `0` otherwise. The model will try to predict this flag using page characteristics (word count, age, title length) *without* looking at the raw CTR or impressions, ensuring it learns the profile of a weak page rather than just doing math on the target.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Filter to our Lane 4 measurable population
measurable = df[(df['impressions_90d'] >= 500) & (df['avg_position'] > 0)].copy()

# Calculate the tier medians and create the proxy label
tier_med = measurable.groupby('position_tier')['ctr'].transform('median')
measurable['is_opportunity'] = (measurable['ctr'] < 0.7 * tier_med).astype(int)

print("Target Class Distribution (Proxy Label):")
print(measurable['is_opportunity'].value_counts(normalize=True).round(3) * 100)
print(f"\nTotal positive cases (opportunities): {measurable['is_opportunity'].sum()}")

Target Class Distribution (Proxy Label):
is_opportunity
0    63.8
1    36.2
Name: proportion, dtype: float64

Total positive cases (opportunities): 6058


## 3. Success metric

*One metric you can defend. What number means 'good'?*



**Primary Metric: Precision@K (specifically, Precision@50).**

An editor only has the capacity to review roughly 50 pages per cycle. Overall accuracy or AUC-ROC across the entire 16,000+ page dataset doesn't matter if the top of the queue is full of false positives (pages that are actually fine). 

If we hand a reviewer the top 50 pages ranked by our model's probability score, Precision@50 tells us exactly how many of those 50 were genuine position-adjusted CTR opportunities. The baseline for a random guess is our positive class rate (~36%). Anything significantly above that proves the model is creating efficiency.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# To demonstrate the baseline metric we must beat:
baseline_precision = measurable['is_opportunity'].mean()

print(f"Random Guess Baseline Precision: {baseline_precision:.1%}")
print(f"If we give a reviewer 50 random visible pages, ~{int(50 * baseline_precision)} will be real opportunities.")
print(f"Our ML model must deliver a Precision@50 > {baseline_precision:.1%} on the unseen test set to be useful.")

Random Guess Baseline Precision: 36.2%
If we give a reviewer 50 random visible pages, ~18 will be real opportunities.
Our ML model must deliver a Precision@50 > 36.2% on the unseen test set to be useful.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*



**One row = One highly-visible, measurable page for a specific client over the last 90 days.**

To ensure the model learns from reliable signal, we drop pages with fewer than 500 impressions or null positions. Critically, we must also drop the leakage columns (`trend_direction`, `trend_pct`, `is_declining_label`) and the exact variables used to create the target (`ctr`, `clicks_90d`) to prevent the model from 'cheating'.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Define columns that would cause target leakage or are direct components of the target
# Define columns that would cause target leakage or are direct components of the target
leakage_and_target_cols = ['trend_direction', 'trend_pct', 'is_declining_label', 'ctr', 'clicks_90d']

# Create the final analytical dataframe
df_analytical = measurable.drop(columns=[col for col in leakage_and_target_cols if col in measurable.columns])

print("Unit of analysis definition:")
print(f"Rows: {df_analytical.shape[0]} measurable pages")
print(f"Features available for ML: {df_analytical.shape[1] - 1} (excluding target)")
print("\nSample of the first 3 rows (target at the end):")

# FIX: Check which columns actually exist to avoid KeyErrors, and use print() instead of display()
desired_cols = ['client_id', 'page_type', 'avg_position', 'is_opportunity']
safe_cols = [col for col in desired_cols if col in df_analytical.columns]

print(df_analytical[safe_cols].head(3))

Unit of analysis definition:
Rows: 16726 measurable pages
Features available for ML: 40 (excluding target)

Sample of the first 3 rows (target at the end):
           client_id  avg_position  is_opportunity
0  client_f369cb89fc          10.6               0
1  client_4e07408562          20.3               1
2  client_7f2253d7e2          36.5               0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*



A fixed rule (`ctr < 0.7 * tier_median`) is perfect for *defining* the target when we have all the data. But to identify *why* a page is failing or to flag pages that are at risk before they bleed clicks for 90 days, we need to look at content features.

An `if-statement` fails here because content variables interact non-linearly. For example, a short title (`title_length < 30`) might be terrible for an informational blog post (low CTR), but perfectly fine for a navigational product page. A single hardcoded rule cannot capture that nuance. A tree-based ML model can map these complex, multi-variable interactions (e.g., `page_type` × `title_length` × `content_age_days`) automatically.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Demonstrate the messiness: checking if a single feature like 'word_count' isolates the target.
# If a simple rule worked, we'd see a massive spike in 'is_opportunity' in one specific bin.
if 'word_count' in df_analytical.columns:
    measurable['word_count_bin'] = pd.qcut(measurable['word_count'], q=4, labels=['Short', 'Medium', 'Long', 'Very Long'])
    
    print("Opportunity Rate by Word Count Quartile:")
    print(measurable.groupby('word_count_bin', observed=True)['is_opportunity'].mean().round(3))
    print("\nThe rates are nearly identical across bins. No single hardcoded rule separates the classes, proving we need ML to find multi-dimensional patterns.")

Opportunity Rate by Word Count Quartile:
word_count_bin
Short        0.367
Medium       0.301
Long         0.295
Very Long    0.338
Name: is_opportunity, dtype: float64

The rates are nearly identical across bins. No single hardcoded rule separates the classes, proving we need ML to find multi-dimensional patterns.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.